# Human Evaluation Sampler

This notebook samples 20 random judged tutor-student turns from one run in `outputs/`, shows the problem and interaction context, and exports a CSV for human evaluation.


## 1. Configuration

Set `OUTPUT_FOLDER` to the run you want to evaluate.


In [ ]:
from pathlib import Path
import json
import random
import pandas as pd
from IPython.display import display, Markdown

OUTPUT_FOLDER = "outputs/20260310_gpt_4_1_x"
SAMPLE_SIZE = 20
RANDOM_SEED = 42
EXPORT_CSV_PATH = "human_evaluation_sample.csv"

DATASET_PATHS = [
    Path("data/LeetCodeDataset-train.csv"),
    Path("data/LeetCodeDataset-test.csv"),
]


## 2. Helpers


In [ ]:
def safe_json_loads(text):
    if isinstance(text, dict):
        return text
    if text is None:
        return {}
    if not isinstance(text, str):
        return {"raw": str(text)}
    text = text.strip()
    if not text:
        return {}
    try:
        return json.loads(text)
    except Exception:
        return {"raw": text}


def list_available_outputs(base_path="outputs"):
    base = Path(base_path)
    if not base.exists():
        return []
    return [
        p.name
        for p in sorted(base.iterdir())
        if p.is_dir() and (p / "interactions").exists()
    ]


def load_problem_lookup(dataset_paths):
    frames = []
    for p in dataset_paths:
        if p.exists():
            df = pd.read_csv(p, usecols=["task_id", "difficulty", "problem_description", "starter_code"])
            frames.append(df)
    if not frames:
        return {}

    all_df = pd.concat(frames, ignore_index=True)
    all_df["starter_code_norm"] = all_df["starter_code"].fillna("").map(lambda s: s.strip())

    lookup = {}
    for _, row in all_df.iterrows():
        key = row["starter_code_norm"]
        if key and key not in lookup:
            lookup[key] = {
                "task_id": row.get("task_id", ""),
                "difficulty": row.get("difficulty", ""),
                "problem_description": row.get("problem_description", ""),
            }
    return lookup


def extract_turn_records(output_folder, problem_lookup):
    output_folder = Path(output_folder)
    interactions_dir = output_folder / "interactions"
    if not interactions_dir.exists():
        raise FileNotFoundError(f"Interactions folder not found: {interactions_dir}")

    records = []

    for interaction_file in sorted(interactions_dir.glob("problem_*.json")):
        with open(interaction_file, "r", encoding="utf-8") as f:
            entries = json.load(f)

        problem_id = interaction_file.stem

        starter_code = ""
        starter_msg = ""
        if entries and "student_message" in entries[0]:
            first_student = entries[0]["student_message"].get("content", "")
            first_payload = safe_json_loads(first_student)
            starter_code = first_payload.get("python_code", "")
            starter_msg = first_payload.get("conversation", "")

        problem_info = problem_lookup.get(starter_code.strip(), {})

        turn_idx = 0
        for entry in entries:
            if not all(k in entry for k in ["tutor_message", "student_message", "tutor_judgment", "student_judgment"]):
                continue

            tutor_message = entry["tutor_message"].get("content", "")
            student_payload = safe_json_loads(entry["student_message"].get("content", ""))
            tutor_judgment = safe_json_loads(entry["tutor_judgment"].get("content", ""))
            student_judgment = safe_json_loads(entry["student_judgment"].get("content", ""))

            record = {
                "run_folder": output_folder.name,
                "problem_id": problem_id,
                "turn_index": turn_idx,
                "task_id": problem_info.get("task_id", ""),
                "difficulty": problem_info.get("difficulty", ""),
                "problem_description": problem_info.get("problem_description", ""),
                "starter_code": starter_code,
                "starter_student_message": starter_msg,
                "tutor_message": tutor_message,
                "student_conversation": student_payload.get("conversation", ""),
                "student_python_code": student_payload.get("python_code", ""),
                "student_level": student_judgment.get("student_level"),
                "student_level_explanation": student_judgment.get("student_level_explanation", ""),
                "student_changed_problem": student_judgment.get("student_changed_problem"),
                "tutor_level": tutor_judgment.get("tutor_level"),
                "tutor_level_explanation": tutor_judgment.get("tutor_level_explanation", ""),
                "leakage_detected": tutor_judgment.get("leakage_detected"),
                "leakage_explanation": tutor_judgment.get("leakage_explanation", ""),
            }

            record["interaction_text"] = (
                f"Tutor:\n{record['tutor_message']}\n\n"
                f"Student:\n{record['student_conversation']}\n\n"
                f"Student code:\n{record['student_python_code']}"
            )

            records.append(record)
            turn_idx += 1

    return records


def sample_records(records, sample_size=20, seed=42):
    if not records:
        return []
    rng = random.Random(seed)
    n = min(sample_size, len(records))
    return rng.sample(records, n)


## 3. Load and Sample


In [ ]:
available = list_available_outputs()
print("Available output folders:")
for i, name in enumerate(available, 1):
    print(f"{i:2d}. {name}")

problem_lookup = load_problem_lookup(DATASET_PATHS)
records = extract_turn_records(OUTPUT_FOLDER, problem_lookup)
sampled_records = sample_records(records, SAMPLE_SIZE, RANDOM_SEED)

print()
print(f"Loaded judged turns: {len(records)}")
print(f"Sampled turns: {len(sampled_records)}")


## 4. Preview in Notebook


In [ ]:
if not sampled_records:
    print("No judged turns found in this output folder.")
else:
    for i, r in enumerate(sampled_records, 1):
        title_line = f"### Sample {i}: {r['problem_id']} | turn {r['turn_index']}"
        meta_line = f"- task_id: {r['task_id'] or 'unknown'} | difficulty: {r['difficulty'] or 'unknown'}"
        levels_line = (
            f"- student_level: {r['student_level']} | tutor_level: {r['tutor_level']} | "
            f"leakage_detected: {r['leakage_detected']}"
        )
        display(Markdown("\n".join([title_line, meta_line, levels_line])))

        if r["problem_description"]:
            display(Markdown("**Problem (truncated):**\n\n" + r["problem_description"][:1200] + "..."))
        else:
            display(Markdown("**Problem:** not found from local dataset lookup (starter code match failed)."))

        display(Markdown("**Interaction:**\n\n```\n" + r["interaction_text"][:3000] + "\n```"))
        display(Markdown("**Student judge explanation:**\n\n" + (r["student_level_explanation"] or "")))
        display(Markdown("**Tutor judge explanation:**\n\n" + (r["tutor_level_explanation"] or "")))


## 5. Export CSV

The CSV includes one row per sampled judged turn.


In [ ]:
export_columns = [
    "run_folder",
    "problem_id",
    "turn_index",
    "task_id",
    "difficulty",
    "problem_description",
    "starter_code",
    "starter_student_message",
    "interaction_text",
    "tutor_message",
    "student_conversation",
    "student_python_code",
    "student_level",
    "student_level_explanation",
    "student_changed_problem",
    "tutor_level",
    "tutor_level_explanation",
    "leakage_detected",
    "leakage_explanation",
]

sample_df = pd.DataFrame(sampled_records)
for col in export_columns:
    if col not in sample_df.columns:
        sample_df[col] = None

sample_df = sample_df[export_columns]
sample_df.to_csv(EXPORT_CSV_PATH, index=False, encoding="utf-8")

print(f"Exported {len(sample_df)} rows to: {EXPORT_CSV_PATH}")
display(sample_df[["problem_id", "turn_index", "task_id", "student_level", "tutor_level", "leakage_detected"]].head(20))
